In [4]:
import cv2
import os
import numpy as np
import pandas as pd
from PIL import Image 
from collections import OrderedDict 
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models
from torchvision.transforms import v2
import random
import seaborn as sns
import torch.nn.functional as F
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
import timm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [5]:
dataset_dir = Path(r"D:\GG\Datasets\Eye Disease Image Dataset\Fourier_Enhanced_Dataset")
image_height = 224
image_width = 224
batch_size = 32
base_learning_rate = 0.0001
epochs = 5
num_workers = os.cpu_count() // 2 if os.cpu_count() else 1 
early_stopping_patience = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using {device} device")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

Using cuda device


In [6]:
# Augmentation
train_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.uint8, scale=True),
    v2.RandomResizedCrop(size=(image_height, image_width), 
                         scale=(0.8, 1.0), ratio=(0.9, 1.1), antialias=True),
    v2.RandomRotation(degrees=15, interpolation=v2.InterpolationMode.BILINEAR, fill=0), 
    v2.RandomHorizontalFlip(p=0.5),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.RandomAdjustSharpness(sharpness_factor=2, p=0.5),
    
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transforms = v2.Compose([
    v2.ToImage(),
    v2.Resize((image_height, image_width), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [8]:
temp_dataset = datasets.ImageFolder(root=dataset_dir)
targets = [s[1] for s in temp_dataset.samples]
dataset_indices = np.arange(len(targets))
class_names = temp_dataset.classes
number_of_classes = len(class_names)
targets = [s[1] for s in temp_dataset.samples]
print(f"Classes: {class_names}")



sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
dev_indices, test_indices = next(sss.split(dataset_indices, targets))


print(f"Total Data: {len(targets)}")
print(f"K-Fold Development Data: {len(dev_indices)}")
print(f"Hold-Out Test Data: {len(test_indices)}")


train_dataset_source = datasets.ImageFolder(root=dataset_dir, transform=train_transforms)
val_test_dataset_source = datasets.ImageFolder(root=dataset_dir, transform=val_test_transforms)


test_dataset = Subset(val_test_dataset_source, test_indices)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=num_workers)


k_folds = 5
kfold = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)


dev_targets = [targets[i] for i in dev_indices]


fold_results = []



for fold, (fold_train_idx_rel, fold_val_idx_rel) in enumerate(kfold.split(dev_indices, dev_targets)):
    

    
    # Map relative indices back to original dataset indices
    train_indices = dev_indices[fold_train_idx_rel]
    val_indices = dev_indices[fold_val_idx_rel]
    
    # Dynamic Class Weights (Calculated per fold) ---
    fold_train_targets = [targets[i] for i in train_indices]
    class_weights_arr = compute_class_weight('balanced', classes=np.unique(fold_train_targets), y=fold_train_targets)
    class_weights = torch.tensor(class_weights_arr, dtype=torch.float).to(device)
    
 
    # Train subset uses train_transforms
    train_subset = Subset(train_dataset_source, train_indices)
    # Val subset uses val_test_transforms
    val_subset = Subset(val_test_dataset_source, val_indices)
    

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    



Classes: ['Color Fundus', 'Diabetic Retinopathy', 'Disc Edema', 'Glaucoma', 'Healthy', 'Macular Scar', 'Myopia', 'Pterygium', 'Retinal Detachment', 'Retinitis Pigmentosa']
Total Data: 16242
K-Fold Development Data: 13805
Hold-Out Test Data: 2437


In [5]:
efficient_model = models.efficientnet_b0(weights=None)
print(efficient_model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [6]:
efficientNetB0 = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
for param in efficientNetB0.features.parameters():
    param.requires_grad = False
for param in efficientNetB0.features[7:].parameters():
    param.requires_grad = True

num_filters = efficientNetB0.classifier[1].in_features
efficientNetB0.classifier[1] = nn.Linear(num_filters, number_of_classes)
efficientNetB0 = efficientNetB0.to(device)
# print("Model details")
# print(efficientNetB0)


criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, efficientNetB0.parameters()), lr = base_learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)



best_val_loss = float('inf') 
patience_counter = 0
save_path_efficient = 'efficientNetB0.pth'
train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    #training the efficientNetB0
    efficientNetB0.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = efficientNetB0(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)


    train_acc = correct / total
    train_loss = running_loss / total
    train_acc_list.append(train_acc)
    train_loss_list.append(train_loss)

    #evaluation
    efficientNetB0.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = efficientNetB0(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)



        val_acc = val_correct / val_total
        val_loss /= val_total
        val_acc_list.append(val_acc)
        val_loss_list.append(val_loss)


        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(efficientNetB0.state_dict(), save_path_efficient)
            print(f"Saved best model (Loss: {best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"No improvement. EarlyStopping counter: {patience_counter}/{early_stopping_patience}")
            if patience_counter >= early_stopping_patience:
                print("Early stopping triggered.")
                break



Epoch 1/50


Epoch 1 validation: 100%|██████████| 77/77 [00:36<00:00,  2.11it/s]


Train Loss: 1.4129, Train Acc: 0.5058
Val Loss: 0.8961, Val Acc: 0.6112
Saved best model (Loss: 0.8961)

Epoch 2/50


Epoch 2 validation: 100%|██████████| 77/77 [00:33<00:00,  2.29it/s]


Train Loss: 0.8271, Train Acc: 0.6581
Val Loss: 0.6752, Val Acc: 0.6929
Saved best model (Loss: 0.6752)

Epoch 3/50


Epoch 3 validation: 100%|██████████| 77/77 [00:33<00:00,  2.31it/s]


... (79 outputs left out)

Epoch 43 validation: 100%|██████████| 77/77 [00:33<00:00,  2.28it/s]


Train Loss: 0.2055, Train Acc: 0.8899
Val Loss: 0.2114, Val Acc: 0.8859
No improvement. EarlyStopping counter: 4/5

Epoch 44/50


Epoch 44 validation: 100%|██████████| 77/77 [00:34<00:00,  2.26it/s]

Train Loss: 0.2003, Train Acc: 0.8927
Val Loss: 0.2067, Val Acc: 0.8879
No improvement. EarlyStopping counter: 5/5
Early stopping triggered.


In [7]:
vgg16_model = models.vgg16(weights=None)
print(vgg16_model)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [8]:
vgg16_model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
for param in vgg16_model.features.parameters():
    param.requires_grad = False

for param in vgg16_model.features[24:].parameters():
    param.requires_grad = True

num_features = vgg16_model.classifier[6].in_features
vgg16_model.classifier[6] = nn.Linear(num_features, number_of_classes) 
vgg16_model = vgg16_model.to(device)
print(f"Model {type(vgg16_model).__name__} initialized for {number_of_classes} global classes.")


criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, vgg16_model.parameters()), lr = base_learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)



best_val_loss = float('inf') 
patience_counter = 0
save_path_vgg16 = 'vgg16.pth'
train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    #training the vgg16_model
    vgg16_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = vgg16_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)


    train_acc = correct / total
    train_loss = running_loss / total
    train_acc_list.append(train_acc)
    train_loss_list.append(train_loss)

    #evaluation
    vgg16_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = vgg16_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)



        val_acc = val_correct / val_total
        val_loss /= val_total
        val_acc_list.append(val_acc)
        val_loss_list.append(val_loss)


        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(vgg16_model.state_dict(), save_path_vgg16)
            print(f"Saved best model (Loss: {best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"No improvement. EarlyStopping counter: {patience_counter}/{early_stopping_patience}")
            if patience_counter >= early_stopping_patience:
                print("Early stopping triggered.")
                break


Model VGG initialized for 10 global classes.

Epoch 1/50


Epoch 1 validation: 100%|██████████| 77/77 [00:34<00:00,  2.24it/s]


Train Loss: 0.9461, Train Acc: 0.6022
Val Loss: 0.6276, Val Acc: 0.6790
Saved best model (Loss: 0.6276)

Epoch 2/50


Epoch 2 validation: 100%|██████████| 77/77 [00:34<00:00,  2.25it/s]


Train Loss: 0.5242, Train Acc: 0.7544
Val Loss: 0.4328, Val Acc: 0.7857
Saved best model (Loss: 0.4328)

Epoch 3/50


Epoch 3 validation: 100%|██████████| 77/77 [00:33<00:00,  2.27it/s]


... (43 outputs left out)

Epoch 25 validation: 100%|██████████| 77/77 [00:34<00:00,  2.24it/s]


Train Loss: 0.0870, Train Acc: 0.9425
Val Loss: 0.1883, Val Acc: 0.9117
No improvement. EarlyStopping counter: 4/5

Epoch 26/50


Epoch 26 validation: 100%|██████████| 77/77 [00:34<00:00,  2.23it/s]

Train Loss: 0.0882, Train Acc: 0.9427
Val Loss: 0.1884, Val Acc: 0.9109
No improvement. EarlyStopping counter: 5/5
Early stopping triggered.


In [9]:
squeezeNet_model = models.squeezenet1_1(weights=None)
print(squeezeNet_model)

SqueezeNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
    (3): Fire(
      (squeeze): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1))
      (squeeze_activation): ReLU(inplace=True)
      (expand1x1): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1))
      (expand1x1_activation): ReLU(inplace=True)
      (expand3x3): Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (expand3x3_activation): ReLU(inplace=True)
    )
    (4): Fire(
      (squeeze): Conv2d(128, 16, kernel_size=(1, 1), stride=(1, 1))
      (squeeze_activation): ReLU(inplace=True)
      (expand1x1): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1))
      (expand1x1_activation): ReLU(inplace=True)
      (expand3x3): Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (expand3x3_activation): ReLU(inplace=True)
    )
    (5): MaxPool2d

In [10]:
squeezeNet_model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.IMAGENET1K_V1)


for param in squeezeNet_model.parameters():
    param.requires_grad = False

for param in squeezeNet_model.features[10:].parameters():
    param.requires_grad = True


original_conv_layer = squeezeNet_model.classifier[1]
num_input_features = original_conv_layer.in_channels 

squeezeNet_model.classifier[1] = nn.Conv2d(
    in_channels=num_input_features,
    out_channels=number_of_classes,
    kernel_size=(1, 1), 
    stride=(1, 1)
)


for param in squeezeNet_model.classifier[1].parameters():
    param.requires_grad = True



squeezeNet_model = squeezeNet_model.to(device)
print(f"Model {type(squeezeNet_model).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, squeezeNet_model.parameters()), lr = base_learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)



best_val_loss = float('inf') 
patience_counter = 0
save_path_squeezeNet = 'squeezeNet_model.pth'
train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    #training the squeezeNet_model
    squeezeNet_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = squeezeNet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)


    train_acc = correct / total
    train_loss = running_loss / total
    train_acc_list.append(train_acc)
    train_loss_list.append(train_loss)

    #evaluation
    squeezeNet_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = squeezeNet_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)



        val_acc = val_correct / val_total
        val_loss /= val_total
        val_acc_list.append(val_acc)
        val_loss_list.append(val_loss)


        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(squeezeNet_model.state_dict(), save_path_squeezeNet)
            print(f"Saved best model (Loss: {best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"No improvement. EarlyStopping counter: {patience_counter}/{early_stopping_patience}")
            if patience_counter >= early_stopping_patience:
                print("Early stopping triggered.")
                break


Model SqueezeNet initialized for 10 global classes.

Epoch 1/50


Epoch 1 validation: 100%|██████████| 77/77 [00:33<00:00,  2.27it/s]


Train Loss: 1.3342, Train Acc: 0.4767
Val Loss: 1.0199, Val Acc: 0.5567
Saved best model (Loss: 1.0199)

Epoch 2/50


Epoch 2 validation: 100%|██████████| 77/77 [00:33<00:00,  2.29it/s]


Train Loss: 0.8762, Train Acc: 0.6329
Val Loss: 0.8224, Val Acc: 0.5998
Saved best model (Loss: 0.8224)

Epoch 3/50


Epoch 3 validation: 100%|██████████| 77/77 [00:33<00:00,  2.31it/s]


... (91 outputs left out)

Epoch 49 validation: 100%|██████████| 77/77 [00:36<00:00,  2.13it/s]


Train Loss: 0.1911, Train Acc: 0.8939
Val Loss: 0.2311, Val Acc: 0.8748
No improvement. EarlyStopping counter: 4/5

Epoch 50/50


Epoch 50 validation: 100%|██████████| 77/77 [00:34<00:00,  2.22it/s]

Train Loss: 0.1933, Train Acc: 0.8935
Val Loss: 0.2311, Val Acc: 0.8752
No improvement. EarlyStopping counter: 5/5
Early stopping triggered.


In [11]:
vit = timm.create_model('vit_base_patch16_224', pretrained=False)
print(vit)

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [12]:
vision_transformer = timm.create_model('vit_base_patch16_224', pretrained=True)


for param in vision_transformer.parameters():
    param.requires_grad = False

for param in vision_transformer.blocks[-1].parameters():
    param.requires_grad = True

num_features = vision_transformer.head.in_features
vision_transformer.head = nn.Linear(num_features, number_of_classes)


vision_transformer = vision_transformer.to(device)
print(f"Model {type(vision_transformer).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, vision_transformer.parameters()), lr = base_learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)



best_val_loss = float('inf') 
patience_counter = 0
save_path_vit = 'vision_transformer.pth'
train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    #training the vision_transformer
    vision_transformer.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = vision_transformer(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)


    train_acc = correct / total
    train_loss = running_loss / total
    train_acc_list.append(train_acc)
    train_loss_list.append(train_loss)

    #evaluation
    vision_transformer.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = vision_transformer(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)



        val_acc = val_correct / val_total
        val_loss /= val_total
        val_acc_list.append(val_acc)
        val_loss_list.append(val_loss)


        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(vision_transformer.state_dict(), save_path_vit)
            print(f"Saved best model (Loss: {best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"No improvement. EarlyStopping counter: {patience_counter}/{early_stopping_patience}")
            if patience_counter >= early_stopping_patience:
                print("Early stopping triggered.")
                break


Model VisionTransformer initialized for 10 global classes.

Epoch 1/50


Epoch 1 validation: 100%|██████████| 77/77 [00:34<00:00,  2.22it/s]


Train Loss: 0.9253, Train Acc: 0.6084
Val Loss: 0.6057, Val Acc: 0.7098
Saved best model (Loss: 0.6057)

Epoch 2/50


Epoch 2 validation: 100%|██████████| 77/77 [00:35<00:00,  2.15it/s]


Train Loss: 0.5071, Train Acc: 0.7615
Val Loss: 0.4516, Val Acc: 0.7886
Saved best model (Loss: 0.4516)

Epoch 3/50


Epoch 3 validation: 100%|██████████| 77/77 [00:35<00:00,  2.16it/s]


... (73 outputs left out)

Epoch 40 validation: 100%|██████████| 77/77 [00:35<00:00,  2.18it/s]


Train Loss: 0.0849, Train Acc: 0.9417
Val Loss: 0.1472, Val Acc: 0.9249
No improvement. EarlyStopping counter: 4/5

Epoch 41/50


Epoch 41 validation: 100%|██████████| 77/77 [00:34<00:00,  2.21it/s]

Train Loss: 0.0827, Train Acc: 0.9453
Val Loss: 0.1473, Val Acc: 0.9257
No improvement. EarlyStopping counter: 5/5
Early stopping triggered.


In [11]:
class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(CustomEnsemble, self).__init__()

        # loading efficientNetB0
        self.efficientNet = models.efficientnet_b0(weights=None)
        efficientNet_features = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(efficientNet_features, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))

        # removing the head to get features
        self.efficientNet.classifier = nn.Identity()

        # loading vgg16
        self.vgg16 = models.vgg16(weights=None)
        vgg16_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg16_features, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))

        self.vgg16.classifier[6] = nn.Identity()

        # loading squeezeNet
        # self.squeezeNet = models.squeezenet1_1(weights=None)
        # self.squeezeNet.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1,1))
        # self.squeezeNet.load_state_dict(torch.load(path_squeeze))


        # loading vision transformer
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_features = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_features, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))

        self.vit.head = nn.Identity()

        # freezing all the layers at first 
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
 

        # Customization
        # unfreezing the last layer of efficientNetB0
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True

        # unfreezing the last layer of vgg16
        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True

        # unfreezing the last layer of vit
        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True        

        # Adding fusion head after adding all features
        total_features = efficientNet_features + vgg16_features + vit_features

        # we have to initialize a neuron layer as a fusion head which will be our classifier
        self.fusion_head = nn.Sequential(
            nn.Linear(total_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        f1 = self.efficientNet(x)
        f2 = self.vgg16(x)
        # f3 = self.squeezeNet(x)
        f4 = self.vit(x)

        concatenated_features = torch.cat((f1, f2, f4), dim=1)

        output = self.fusion_head(concatenated_features)
        return output
    


ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    # path_squeeze='squeezeNet_model.pth',
    path_vit='vision_transformer.pth'
)

ensemble_model = ensemble_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, ensemble_model.parameters()), lr=base_learning_rate, weight_decay=1e-3)        
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)




Epoch 1/20


Epoch 1 Validation: 100%|██████████| 77/77 [00:38<00:00,  1.99it/s]


Train Loss: 0.3566, Train Acc: 0.8952
Val Loss: 0.2003, Val Acc: 0.9113

Epoch 2/20


Epoch 2 Validation: 100%|██████████| 77/77 [00:35<00:00,  2.20it/s]


Train Loss: 0.1892, Train Acc: 0.9193
Val Loss: 0.1899, Val Acc: 0.9117

Epoch 3/20


Epoch 3 Validation: 100%|██████████| 77/77 [00:37<00:00,  2.04it/s]


... (31 outputs left out)

Epoch 19 Validation: 100%|██████████| 77/77 [00:35<00:00,  2.17it/s]


Train Loss: 0.1019, Train Acc: 0.9434
Val Loss: 0.1423, Val Acc: 0.9290

Epoch 20/20


Epoch 20 Validation: 100%|██████████| 77/77 [00:35<00:00,  2.19it/s]

Train Loss: 0.1010, Train Acc: 0.9428
Val Loss: 0.1431, Val Acc: 0.9269


In [50]:
import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from torch.utils.data import Subset, DataLoader
from sklearn.utils.class_weight import compute_class_weight

# -----------------------------------------------------------------------
# 1. PREPARE DATASET & HOLD-OUT TEST SET
# -----------------------------------------------------------------------
temp_dataset = datasets.ImageFolder(root=dataset_dir)
targets = [s[1] for s in temp_dataset.samples]
dataset_indices = np.arange(len(targets))
class_names = temp_dataset.classes
number_of_classes = len(class_names)
targets = [s[1] for s in temp_dataset.samples]
print(f"Classes: {class_names}")


# --- Step A: Set aside 15% strictly for FINAL TESTING (The Hold-Out Set) ---
# We use StratifiedShuffleSplit once to isolate the Test Set
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
dev_indices, test_indices = next(sss.split(dataset_indices, targets))

# 'dev_indices' contains the 85% we will use for K-Fold (Train + Val)
# 'test_indices' contains the 15% we lock away

print(f"Total Data: {len(targets)}")
print(f"K-Fold Development Data: {len(dev_indices)}")
print(f"Hold-Out Test Data: {len(test_indices)}")

# Prepare the Source Datasets (One for Train transforms, one for Val/Test transforms)
train_dataset_source = datasets.ImageFolder(root=dataset_dir, transform=train_transforms)
val_test_dataset_source = datasets.ImageFolder(root=dataset_dir, transform=val_test_transforms)

# The Test Loader is constant and never changes
test_dataset = Subset(val_test_dataset_source, test_indices)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=num_workers)

# -----------------------------------------------------------------------
# 2. STRATIFIED K-FOLD LOOP
# -----------------------------------------------------------------------
k_folds = 5
kfold = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

# We need the targets specifically for the development set to stratify the folds
dev_targets = [targets[i] for i in dev_indices]

# Store results
fold_results = []

print(f"\nStarting {k_folds}-Fold Cross-Validation...")

# Loop through the folds
for fold, (fold_train_idx_rel, fold_val_idx_rel) in enumerate(kfold.split(dev_indices, dev_targets)):
    
    print(f"\n{'='*20} Fold {fold+1}/{k_folds} {'='*20}")
    
    # Map relative indices back to original dataset indices
    train_indices = dev_indices[fold_train_idx_rel]
    val_indices = dev_indices[fold_val_idx_rel]
    
    # --- Dynamic Class Weights (Calculated per fold) ---
    fold_train_targets = [targets[i] for i in train_indices]
    class_weights_arr = compute_class_weight('balanced', classes=np.unique(fold_train_targets), y=fold_train_targets)
    class_weights = torch.tensor(class_weights_arr, dtype=torch.float).to(device)
    
    # --- Create Subsets ---
    # Train subset uses train_transforms
    train_subset = Subset(train_dataset_source, train_indices)
    # Val subset uses val_test_transforms
    val_subset = Subset(val_test_dataset_source, val_indices)
    
    # --- Create Loaders ---
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    


Classes: ['Color Fundus', 'Diabetic Retinopathy', 'Disc Edema', 'Glaucoma', 'Healthy', 'Macular Scar', 'Myopia', 'Retinal Detachment', 'Retinitis Pigmentosa']
Total Data: 16140
K-Fold Development Data: 13719
Hold-Out Test Data: 2421

Starting 5-Fold Cross-Validation...

==================== Fold 1/5 ====================

==================== Fold 2/5 ====================

==================== Fold 3/5 ====================

==================== Fold 4/5 ====================

==================== Fold 5/5 ====================


In [21]:
class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit, temperature=2.0):
        super(CustomEnsemble, self).__init__()
        self.temperature = temperature  # For knowledge calibration
        
        # ============= 1. LOAD PRETRAINED MODELS =============
        
        # EfficientNet-B0
        self.efficientNet = models.efficientnet_b0(weights=None)
        efficientNet_features = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(efficientNet_features, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))
        self.efficientNet.classifier = nn.Identity()
        
        # VGG16
        self.vgg16 = models.vgg16(weights=None)
        vgg16_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg16_features, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))
        self.vgg16.classifier[6] = nn.Identity()
        
        # Vision Transformer
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_features = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_features, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))
        self.vit.head = nn.Identity()
        
        # Freeze base models
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
        
        # Unfreeze last layers
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True
        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True
        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True
        
        # ============= 2. FEATURE NORMALIZATION =============
        # Each model's features are normalized before fusion
        self.eff_norm = nn.LayerNorm(efficientNet_features)
        self.vgg_norm = nn.LayerNorm(vgg16_features)
        self.vit_norm = nn.LayerNorm(vit_features)
        
        # Optional: Bottleneck layers to reduce dimensions
        self.eff_bottleneck = nn.Linear(efficientNet_features, 512)
        self.vgg_bottleneck = nn.Linear(vgg16_features, 512)
        self.vit_bottleneck = nn.Linear(vit_features, 512)
        
        # ============= 3. ATTENTION-BASED FUSION =============
        # Learn which model to trust per sample/image
        total_features = 512 * 3  # After bottleneck reduction
        
        # Gating network - learns model importance per image
        self.gating_network = nn.Sequential(
            nn.Linear(total_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 3),  # 3 models
            nn.Softmax(dim=1)
        )
        
        # ============= 4. FUSION HEAD =============
        self.fusion_head = nn.Sequential(
            nn.Linear(total_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
        
    def forward(self, x):
        # Extract features
        f1 = self.efficientNet(x)  # [batch, 1280]
        f2 = self.vgg16(x)          # [batch, 4096]
        f4 = self.vit(x)            # [batch, 768]
        
        # ============= FEATURE NORMALIZATION =============
        f1_norm = self.eff_norm(f1)
        f2_norm = self.vgg_norm(f2)
        f4_norm = self.vit_norm(f4)
        
        # ============= BOTTLENECK (Optional but recommended) =============
        f1_proj = self.eff_bottleneck(f1_norm)
        f2_proj = self.vgg_bottleneck(f2_norm)
        f4_proj = self.vit_bottleneck(f4_norm)
        
        # ============= ATTENTION-BASED FUSION =============
        # Create combined features for gating
        combined_for_gate = torch.cat([f1_proj, f2_proj, f4_proj], dim=1)
        
        # Get attention weights (which model to trust)
        gate_weights = self.gating_network(combined_for_gate)  # [batch, 3]
        
        # Apply attention weights to each model's features
        f1_weighted = f1_proj * gate_weights[:, 0:1]
        f2_weighted = f2_proj * gate_weights[:, 1:2]
        f4_weighted = f4_proj * gate_weights[:, 2:3]
        
        # Concatenate weighted features
        concatenated_features = torch.cat([f1_weighted, f2_weighted, f4_weighted], dim=1)
        
        # Final classification
        output = self.fusion_head(concatenated_features)
        
        return output
    

def test_time_augmentation(model, image, device):
    """
    Apply TTA to a single image
    """
    model.eval()
    
    # Define augmentations for TTA
    tta_transforms = [
        lambda x: x,  # Original
        lambda x: torch.flip(x, dims=[3]),  # Horizontal flip
        lambda x: torch.rot90(x, 1, dims=[2, 3]),  # Rotate 90
        lambda x: torch.rot90(x, 2, dims=[2, 3]),  # Rotate 180
        lambda x: torch.rot90(x, 3, dims=[2, 3]),  # Rotate 270
    ]
    
    all_outputs = []
    
    with torch.no_grad():
        for transform in tta_transforms:
            augmented_image = transform(image)
            output = model(augmented_image)
            
            # Apply inverse transform if needed for consistency
            all_outputs.append(output)
    
    # Average predictions
    return torch.mean(torch.stack(all_outputs), dim=0)

# # In your test loop:
# with torch.no_grad():
#     for images, labels in test_loader:
#         images = images.to(device)
        
#         # Apply TTA to each image
#         for i in range(images.size(0)):
#             single_image = images[i:i+1]
#             probs = test_time_augmentation(ensemble_model, single_image, device)
#             # ... rest of evaluation


def evaluate_with_tta(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="TTA Evaluation"):
            images = images.to(device)
            labels = labels.to(device)
            
            batch_preds = []
            for i in range(images.size(0)):
                single_image = images[i:i+1]
                # Apply TTA to this image
                tta_transforms = [
                    lambda x: x,
                    lambda x: torch.flip(x, dims=[3]),
                    lambda x: torch.rot90(x, 1, [2, 3]),
                    lambda x: torch.rot90(x, 2, [2, 3]),
                ]
                aug_outputs = []
                for aug in tta_transforms:
                    aug_img = aug(single_image)
                    output = model(aug_img)
                    aug_outputs.append(output)
                avg_output = torch.mean(torch.stack(aug_outputs), dim=0)
                batch_preds.append(avg_output)
            
            batch_preds = torch.cat(batch_preds, dim=0)
            _, preds = torch.max(batch_preds, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"✅ TTA Ensemble Accuracy: {accuracy:.4f}")
    return accuracy




def analyze_gate_weights(model, val_loader, device):
    model.eval()
    all_weights = []
    with torch.no_grad():
        for images, _ in val_loader:
            images = images.to(device)
            f1 = model.efficientNet(images)
            f2 = model.vgg16(images)
            f4 = model.vit(images)
            
            f1_proj = model.eff_bottleneck(model.eff_norm(f1))
            f2_proj = model.vgg_bottleneck(model.vgg_norm(f2))
            f4_proj = model.vit_bottleneck(model.vit_norm(f4))
            
            combined = torch.cat([f1_proj, f2_proj, f4_proj], dim=1)
            gate_weights = model.gating_network(combined)
            all_weights.append(gate_weights.cpu())
    
    gate_weights = torch.cat(all_weights)
    print(f"\n[Gate Weights] EfficientNet: {gate_weights[:,0].mean():.3f} ± {gate_weights[:,0].std():.3f}")
    print(f"[Gate Weights] VGG16:        {gate_weights[:,1].mean():.3f} ± {gate_weights[:,1].std():.3f}")
    print(f"[Gate Weights] ViT:          {gate_weights[:,2].mean():.3f} ± {gate_weights[:,2].std():.3f}")
    return gate_weights

class TemperatureScaledEnsemble(nn.Module):
    def __init__(self, ensemble_model, temperature=2.0):
        super().__init__()
        self.ensemble_model = ensemble_model
        self.temperature = nn.Parameter(torch.ones(1) * temperature)
        
    def forward(self, x):
        logits = self.ensemble_model(x)
        return logits / self.temperature.to(logits.device)  # <-- FIX
    
    def calibrate_temperature(self, val_loader, device):
        self.eval()
        logits_list, labels_list = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                logits = self.ensemble_model(images)
                logits_list.append(logits.cpu())
                labels_list.append(labels)
        
        logits = torch.cat(logits_list)
        labels = torch.cat(labels_list)
        
        # Move temperature to CPU for optimization
        self.temperature = nn.Parameter(self.temperature.to('cpu'))
        optimizer = optim.LBFGS([self.temperature], lr=0.01, max_iter=50)
        
        def eval_loss():
            optimizer.zero_grad()
            scaled_logits = logits / self.temperature
            loss = F.cross_entropy(scaled_logits, labels)
            loss.backward()
            return loss
        
        optimizer.step(eval_loss)
        print(f"Optimal temperature: {self.temperature.item():.3f}")
        # Move back to original device
        self.temperature = nn.Parameter(self.temperature.to(device))




# ============= 1. CREATE ENSEMBLE =============
ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    path_vit='vision_transformer.pth'
).to(device)

# ============= 2. VERIFY TRAINABLE PARAMETERS =============
trainable_params = []
for name, param in ensemble_model.named_parameters():
    if param.requires_grad:
        trainable_params.append(param)
        print(f"✓ Trainable: {name} - {param.shape}")
print(f"Total trainable parameters: {len(trainable_params)}")

# ============= 3. OPTIMIZER WITH DIFFERENT LEARNING RATES =============
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW([
    # Backbone fine-tuning (very low LR)
    {'params': ensemble_model.efficientNet.features[7:].parameters(), 'lr': 1e-5},
    {'params': ensemble_model.vgg16.features[24:].parameters(), 'lr': 1e-5},
    {'params': ensemble_model.vit.blocks[-1].parameters(), 'lr': 1e-5},
    
    # Feature normalization (moderate LR)
    {'params': ensemble_model.eff_norm.parameters(), 'lr': 1e-3},
    {'params': ensemble_model.vgg_norm.parameters(), 'lr': 1e-3},
    {'params': ensemble_model.vit_norm.parameters(), 'lr': 1e-3},
    
    # Bottleneck layers (high LR)
    {'params': ensemble_model.eff_bottleneck.parameters(), 'lr': 1e-3},
    {'params': ensemble_model.vgg_bottleneck.parameters(), 'lr': 1e-3},
    {'params': ensemble_model.vit_bottleneck.parameters(), 'lr': 1e-3},
    
    # Gating network (high LR)
    {'params': ensemble_model.gating_network.parameters(), 'lr': 1e-3},
    
    # Fusion head (high LR)
    {'params': ensemble_model.fusion_head.parameters(), 'lr': 1e-3},
], weight_decay=1e-4)

# Cosine annealing with warm restarts - much better for this task
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# ============= 4. TRAINING LOOP (NO TEMPERATURE SCALING) =============
best_val_acc = 0.0
best_epoch = 0
patience_counter = 0
early_stopping_patience = 10
epochs = 50  # You need this many!

for epoch in range(epochs):
    # ----- TRAINING -----
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = ensemble_model(images)  # <-- NO temperature scaling!
        loss = criterion(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(ensemble_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    train_acc = correct / total
    train_loss = running_loss / total
    
    # ----- VALIDATION -----
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    val_acc = val_correct / val_total
    val_loss /= val_total
    
    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': ensemble_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            'val_loss': val_loss,
            'train_acc': train_acc,
        }, 'best_ensemble_improved.pth')
        print(f"✓ Saved best model (epoch {epoch+1}, val_acc: {val_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= early_stopping_patience:
            print(f"✓ Early stopping at epoch {epoch+1}. Best epoch: {best_epoch}")
            break
    
    # ----- DIAGNOSTICS (every 5 epochs) -----
    if (epoch + 1) % 5 == 0:
        analyze_gate_weights(ensemble_model, val_loader, device)
    
    print(f"Epoch {epoch+1}: Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, LR: {current_lr:.2e}")

# ============= 5. POST-TRAINING: LOAD BEST MODEL, CALIBRATE TEMPERATURE, TTA =============
checkpoint = torch.load('best_ensemble_improved.pth')
ensemble_model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1} with val_acc: {checkpoint['best_val_acc']:.4f}")

# Create temperature-scaled version for inference
temperature_model = TemperatureScaledEnsemble(ensemble_model).to(device)
temperature_model.calibrate_temperature(val_loader, device)

# Evaluate with TTA
final_accuracy = evaluate_with_tta(temperature_model, test_loader, device)

✓ Trainable: efficientNet.features.7.0.block.0.0.weight - torch.Size([1152, 192, 1, 1])
✓ Trainable: efficientNet.features.7.0.block.0.1.weight - torch.Size([1152])
✓ Trainable: efficientNet.features.7.0.block.0.1.bias - torch.Size([1152])
✓ Trainable: efficientNet.features.7.0.block.1.0.weight - torch.Size([1152, 1, 3, 3])
✓ Trainable: efficientNet.features.7.0.block.1.1.weight - torch.Size([1152])
✓ Trainable: efficientNet.features.7.0.block.1.1.bias - torch.Size([1152])
✓ Trainable: efficientNet.features.7.0.block.2.fc1.weight - torch.Size([48, 1152, 1, 1])
✓ Trainable: efficientNet.features.7.0.block.2.fc1.bias - torch.Size([48])
✓ Trainable: efficientNet.features.7.0.block.2.fc2.weight - torch.Size([1152, 48, 1, 1])
✓ Trainable: efficientNet.features.7.0.block.2.fc2.bias - torch.Size([1152])
✓ Trainable: efficientNet.features.7.0.block.3.0.weight - torch.Size([320, 1152, 1, 1])
✓ Trainable: efficientNet.features.7.0.block.3.1.weight - torch.Size([320])
✓ Trainable: efficientNet.fe

Epoch 1: 100%|██████████| 346/346 [02:00<00:00,  2.88it/s]


✓ Saved best model (epoch 1, val_acc: 0.9113)
Epoch 1: Train Acc: 0.9127, Val Acc: 0.9113, LR: 9.78e-06


Epoch 2: 100%|██████████| 346/346 [01:55<00:00,  2.99it/s]


✓ Saved best model (epoch 2, val_acc: 0.9232)
Epoch 2: Train Acc: 0.9239, Val Acc: 0.9232, LR: 9.14e-06


Epoch 3: 100%|██████████| 346/346 [01:53<00:00,  3.04it/s]


... (21 outputs left out)

d:\Practise\venv\Lib\site-packages\torch\optim\lbfgs.py:457: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:836.)
  loss = float(closure())


Optimal temperature: 2.106


TTA Evaluation: 100%|██████████| 2437/2437 [02:35<00:00, 15.71it/s]

✅ TTA Ensemble Accuracy: 0.8847


In [28]:
class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super().__init__()
        # ---------- LOAD BACKBONES (IDENTICAL TO YOUR BASELINE) ----------
        self.efficientNet = models.efficientnet_b0(weights=None)
        efficientNet_features = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(efficientNet_features, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))
        self.efficientNet.classifier = nn.Identity()

        self.vgg16 = models.vgg16(weights=None)
        vgg16_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg16_features, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))
        self.vgg16.classifier[6] = nn.Identity()

        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_features = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_features, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))
        self.vit.head = nn.Identity()

        # ---------- FREEZE / UNFREEZE (SAME AS BASELINE) ----------
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
        for param in self.efficientNet.features[7:].parameters(): param.requires_grad = True
        for param in self.vgg16.features[24:].parameters(): param.requires_grad = True
        for param in self.vit.blocks[-1].parameters(): param.requires_grad = True

        # ---------- BOTTLENECK (NO LAYER NORM) ----------
        bottleneck_dim = 512
        self.eff_bottleneck = nn.Linear(efficientNet_features, bottleneck_dim)
        self.vgg_bottleneck = nn.Linear(vgg16_features, bottleneck_dim)
        self.vit_bottleneck = nn.Linear(vit_features, bottleneck_dim)

        # ---------- FUSION HEAD (INPUT = bottleneck_dim * 3) ----------
        self.fusion_head = nn.Sequential(
            nn.Linear(bottleneck_dim * 3, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # Extract features ONCE
        f1 = self.efficientNet(x)
        f2 = self.vgg16(x)
        f3 = self.vit(x)

        # Bottleneck projections
        f1 = self.eff_bottleneck(f1)
        f2 = self.vgg_bottleneck(f2)
        f3 = self.vit_bottleneck(f3)

        concat = torch.cat([f1, f2, f3], dim=1)
        return self.fusion_head(concat)

ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    # path_squeeze='squeezeNet_model.pth',
    path_vit='vision_transformer.pth'
)

ensemble_model = ensemble_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW([
    # Backbones (very low LR)
    {'params': ensemble_model.efficientNet.features[7:].parameters(), 'lr': 1e-5},
    {'params': ensemble_model.vgg16.features[24:].parameters(), 'lr': 1e-5},
    {'params': ensemble_model.vit.blocks[-1].parameters(), 'lr': 1e-5},
    
    # Bottlenecks (HIGH LR – randomly initialized)
    {'params': ensemble_model.eff_bottleneck.parameters(), 'lr': 1e-3},
    {'params': ensemble_model.vgg_bottleneck.parameters(), 'lr': 1e-3},
    {'params': ensemble_model.vit_bottleneck.parameters(), 'lr': 1e-3},
    
    # Fusion head (HIGH LR)
    {'params': ensemble_model.fusion_head.parameters(), 'lr': 1e-3},
], weight_decay=1e-4)      
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)




Epoch 1/20


Epoch 1 Validation: 100%|██████████| 87/87 [00:39<00:00,  2.18it/s]


Train Loss: 0.2270, Train Acc: 0.9186
Val Loss: 0.1394, Val Acc: 0.9272

Epoch 2/20


Epoch 2 Validation: 100%|██████████| 87/87 [00:40<00:00,  2.16it/s]


Train Loss: 0.1566, Train Acc: 0.9278
Val Loss: 0.1269, Val Acc: 0.9294

Epoch 3/20


Epoch 3 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.11it/s]


... (31 outputs left out)

Epoch 19 Validation: 100%|██████████| 87/87 [00:39<00:00,  2.18it/s]


Train Loss: 0.0878, Train Acc: 0.9433
Val Loss: 0.1135, Val Acc: 0.9247

Epoch 20/20


Epoch 20 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.07it/s]

Train Loss: 0.0839, Train Acc: 0.9440
Val Loss: 0.1164, Val Acc: 0.9225


### Using Squueze-excitation

In [16]:
class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(CustomEnsemble, self).__init__()

        # --- 1. Load EfficientNet ---
        self.efficientNet = models.efficientnet_b0(weights=None)
        eff_in = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(eff_in, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))
        self.efficientNet.classifier = nn.Identity() # Output: 1280

        # --- 2. Load VGG16 ---
        self.vgg16 = models.vgg16(weights=None)
        vgg_in = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg_in, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))
        self.vgg16.classifier[6] = nn.Identity() # Output: 4096

        # --- 3. Load ViT ---
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_in = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_in, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))
        self.vit.head = nn.Identity() # Output: 768

        # --- 4. Freeze All First ---
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
 
        # --- 5. Unfreeze Specific Layers (Fine Tuning) ---
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True
        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True
        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True        

        # --- 6. Custom Heads (Projectors) ---
        # These compress the varying sizes to a standard 256
        self.eff_head = nn.Sequential(
            nn.Linear(1280, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.vgg_head = nn.Sequential(
            nn.Linear(4096, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.vit_head = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # --- 7. Fusion Head ---
        # [CRITICAL FIX]: The input is now 256 + 256 + 256, NOT the original sizes
        fusion_input_dim = 256 * 3 

        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256), # Added an extra layer for depth
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Get raw features
        f1 = self.efficientNet(x)
        f2 = self.vgg16(x)
        f3 = self.vit(x)

        # Project to 256 dim each
        f1 = self.eff_head(f1)
        f2 = self.vgg_head(f2)
        f3 = self.vit_head(f3)

        # Concatenate (Size: 768)
        concatenated_features = torch.cat([f1, f2, f3], dim=1)
        
        # Classify
        output = self.fusion_head(concatenated_features)
        
        return output

# Initialize Model
ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    path_vit='vision_transformer.pth'
)
ensemble_model = ensemble_model.to(device)

# ==============================================================================
# DIFFERENTIAL LEARNING RATES IMPLEMENTATION
# ==============================================================================

# 1. Group the parameters
backbone_params = []
head_params = []

# Add Backbones (Only the ones with requires_grad=True will be updated)
backbone_params += [p for p in ensemble_model.efficientNet.parameters() if p.requires_grad]
backbone_params += [p for p in ensemble_model.vgg16.parameters() if p.requires_grad]
backbone_params += [p for p in ensemble_model.vit.parameters() if p.requires_grad]

# Add New Heads (All of these require grad by default)
head_params += list(ensemble_model.eff_head.parameters())
head_params += list(ensemble_model.vgg_head.parameters())
head_params += list(ensemble_model.vit_head.parameters())
head_params += list(ensemble_model.fusion_head.parameters())

# 2. Define Optimizer with Groups
optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5}, # Low LR for pre-trained parts
    {'params': head_params, 'lr': 5e-4}      # Higher LR for new random parts
], weight_decay=1e-3)

# Scheduler (Standard)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

print(f"Optimizer initialized: Backbones (lr=1e-5), Heads (lr=5e-4)")

# ==============================================================================
# TRAINING LOOP (Remains the same)
# ==============================================================================

train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)

Optimizer initialized: Backbones (lr=1e-5), Heads (lr=5e-4)

Epoch 1/20


Epoch 1 Validation: 100%|██████████| 87/87 [00:46<00:00,  1.87it/s]


Train Loss: 0.3679, Train Acc: 0.8883
Val Loss: 0.1506, Val Acc: 0.9283

Epoch 2/20


Epoch 2 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.05it/s]


Train Loss: 0.1686, Train Acc: 0.9278
Val Loss: 0.1320, Val Acc: 0.9294

Epoch 3/20


Epoch 3 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.05it/s]


... (31 outputs left out)

Epoch 19 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.08it/s]


Train Loss: 0.0944, Train Acc: 0.9443
Val Loss: 0.1169, Val Acc: 0.9272

Epoch 20/20


Epoch 20 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.07it/s]

Train Loss: 0.0953, Train Acc: 0.9414
Val Loss: 0.1148, Val Acc: 0.9279


In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import timm
from tqdm import tqdm

class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(CustomEnsemble, self).__init__()

        # --- 1. Load EfficientNetB0 ---
        self.efficientNet = models.efficientnet_b0(weights=None)
        efficientNet_features = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(efficientNet_features, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))
        self.efficientNet.classifier = nn.Identity() # Output: 1280

        # --- 2. Load VGG16 ---
        self.vgg16 = models.vgg16(weights=None)
        vgg16_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg16_features, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))
        self.vgg16.classifier[6] = nn.Identity() # Output: 4096

        # --- 3. Load Vision Transformer ---
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_features = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_features, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))
        self.vit.head = nn.Identity() # Output: 768

        # --- 4. Freeze Everything Initially ---
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
 
        # --- 5. Unfreeze Specific Layers ---
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True

        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True

        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True        

        total_features = efficientNet_features + vgg16_features + vit_features

        # ============================================================
        # NEW: Gated Attention Mechanism
        # ============================================================
        # Inputs: The total concatenated features
        # Outputs: 3 weights (one for EfficientNet, VGG, ViT)
        self.attention_gate = nn.Sequential(
            nn.Linear(total_features, 512),
            nn.ReLU(),
            nn.Linear(512, 3), # 3 numbers: w1, w2, w3
            nn.Softmax(dim=1)  # Ensures weights sum to 1 (e.g., 0.2 + 0.7 + 0.1 = 1.0)
        )

        # Fusion Head (Same as your code)
        self.fusion_head = nn.Sequential(
            nn.Linear(total_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        f1 = self.efficientNet(x) # 1280
        f2 = self.vgg16(x)        # 4096
        f4 = self.vit(x)          # 768

        # 1. Concatenate raw features to calculate attention weights
        raw_concatenated = torch.cat((f1, f2, f4), dim=1)
        
        # 2. Get Weights [Batch_Size, 3]
        weights = self.attention_gate(raw_concatenated)
        
        # 3. Reshape weights for multiplication: [Batch_Size, 1]
        w1 = weights[:, 0].unsqueeze(1)
        w2 = weights[:, 1].unsqueeze(1)
        w3 = weights[:, 2].unsqueeze(1)

        # 4. Multiply original features by their learned importance
        f1_weighted = f1 * w1
        f2_weighted = f2 * w2
        f4_weighted = f4 * w3

        # 5. Concatenate the WEIGHTED features
        weighted_concatenated = torch.cat((f1_weighted, f2_weighted, f4_weighted), dim=1)

        # 6. Final Classification
        output = self.fusion_head(weighted_concatenated)
        return output
    

# ============================================================
# Main Execution Logic
# ============================================================

ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    path_vit='vision_transformer.pth'
)

ensemble_model = ensemble_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Note: The filter() here automatically includes the new attention_gate parameters
# because they are initialized with requires_grad=True by default.
optimizer = optim.Adam(filter(lambda p: p.requires_grad, ensemble_model.parameters()), 
                       lr=base_learning_rate, 
                       weight_decay=1e-3)        

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)


Epoch 1/20


Epoch 1 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.06it/s]


Train Loss: 0.3712, Train Acc: 0.8892
Val Loss: 0.1869, Val Acc: 0.9167

Epoch 2/20


Epoch 2 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.04it/s]


Train Loss: 0.1922, Train Acc: 0.9230
Val Loss: 0.1872, Val Acc: 0.9076

Epoch 3/20


Epoch 3 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.08it/s]


... (31 outputs left out)

Epoch 19 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.07it/s]


Train Loss: 0.1067, Train Acc: 0.9439
Val Loss: 0.1502, Val Acc: 0.9192

Epoch 20/20


Epoch 20 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.08it/s]

Train Loss: 0.1069, Train Acc: 0.9444
Val Loss: 0.1543, Val Acc: 0.9131


In [18]:
# ==============================================================================
# CODE BASE 2: Gated Attention Ensemble
# ==============================================================================

class EnsembleGatedAttention(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(EnsembleGatedAttention, self).__init__()

        # --- 1. Load EfficientNet ---
        self.efficientNet = models.efficientnet_b0(weights=None)
        eff_in = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(eff_in, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))
        self.efficientNet.classifier = nn.Identity()

        # --- 2. Load VGG16 ---
        self.vgg16 = models.vgg16(weights=None)
        vgg_in = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg_in, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))
        self.vgg16.classifier[6] = nn.Identity()

        # --- 3. Load ViT ---
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_in = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_in, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))
        self.vit.head = nn.Identity()

        # --- 4. Freeze All First ---
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
 
        # --- 5. Unfreeze Specific Layers ---
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True
        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True
        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True        

        # --- 6. Projectors (Map to 256) ---
        self.eff_head = nn.Sequential(
            nn.Linear(1280, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3)
        )
        self.vgg_head = nn.Sequential(
            nn.Linear(4096, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3)
        )
        self.vit_head = nn.Sequential(
            nn.Linear(768, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3)
        )

        # --- 7. Gated Attention Mechanism ---
        # Takes concatenated projected features (256*3=768) and predicts 3 weights
        self.attention_gate = nn.Sequential(
            nn.Linear(256 * 3, 128),
            nn.ReLU(),
            nn.Linear(128, 3), # Output 3 scores
            nn.Softmax(dim=1)  # Normalize scores to sum to 1
        )

        # --- 8. Fusion Head ---
        # Input is still 768 because we concatenate the WEIGHTED features
        self.fusion_head = nn.Sequential(
            nn.Linear(256 * 3, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # 1. Get raw features
        f1 = self.efficientNet(x)
        f2 = self.vgg16(x)
        f3 = self.vit(x)

        # 2. Project to 256 dim
        p1 = self.eff_head(f1)
        p2 = self.vgg_head(f2)
        p3 = self.vit_head(f3)

        # 3. Concatenate to calculate attention
        concat = torch.cat([p1, p2, p3], dim=1) # Shape: [Batch, 768]
        
        # 4. Calculate Weights
        weights = self.attention_gate(concat) # Shape: [Batch, 3] e.g., [0.2, 0.7, 0.1]
        
        # 5. Apply weights
        # We need to reshape weights to multiply properly: [Batch, 1]
        w1 = weights[:, 0].unsqueeze(1)
        w2 = weights[:, 1].unsqueeze(1)
        w3 = weights[:, 2].unsqueeze(1)

        # Multiply projected features by their importance
        weighted_p1 = p1 * w1
        weighted_p2 = p2 * w2
        weighted_p3 = p3 * w3

        # 6. Re-concatenate weighted features
        final_concat = torch.cat([weighted_p1, weighted_p2, weighted_p3], dim=1)

        # 7. Classify
        output = self.fusion_head(final_concat)
        return output

# --- Initialize ---
ensemble_model = EnsembleGatedAttention(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    path_vit='vision_transformer.pth'
)
ensemble_model = ensemble_model.to(device)

# --- Differential Learning Rates ---
backbone_params = []
backbone_params += [p for p in ensemble_model.efficientNet.parameters() if p.requires_grad]
backbone_params += [p for p in ensemble_model.vgg16.parameters() if p.requires_grad]
backbone_params += [p for p in ensemble_model.vit.parameters() if p.requires_grad]

head_params = []
head_params += list(ensemble_model.eff_head.parameters())
head_params += list(ensemble_model.vgg_head.parameters())
head_params += list(ensemble_model.vit_head.parameters())
# Add attention gate to head params
head_params += list(ensemble_model.attention_gate.parameters()) 
head_params += list(ensemble_model.fusion_head.parameters())

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5}, 
    {'params': head_params, 'lr': 5e-4}      
], weight_decay=1e-3)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

# --- Training Loop ---
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs} [Gated Attention]")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x0000028F96ABE7A0>
Traceback (most recent call last):
  File "d:\Practise\venv\Lib\site-packages\torch\utils\data\dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "d:\Practise\venv\Lib\site-packages\torch\utils\data\dataloader.py", line 1622, in _shutdown_workers
    if self._persistent_workers or self._workers_status[worker_id]:
                                   ^^^^^^^^^^^^^^^^^^^^
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x0000028F96ABE7A0>
Traceback (most recent call last):
  File "d:\Practise\venv\Lib\site-packages\torch\utils\data\dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "d:\Practise\venv\Lib\site-packages\torch\utils\data\dataloader.py", line 1622, in _shutdown_workers
    if self._persistent_workers or self._workers_status[w


Epoch 1/20 [Gated Attention]


Validation: 100%|██████████| 87/87 [00:42<00:00,  2.06it/s]


Train Loss: 0.2285, Train Acc: 0.9124
Val Loss: 0.1291, Val Acc: 0.9301

Epoch 2/20 [Gated Attention]


Validation: 100%|██████████| 87/87 [00:42<00:00,  2.06it/s]


Train Loss: 0.1360, Train Acc: 0.9344
Val Loss: 0.1236, Val Acc: 0.9290

Epoch 3/20 [Gated Attention]


... (32 outputs left out)

Validation: 100%|██████████| 87/87 [00:40<00:00,  2.14it/s]


Train Loss: 0.0818, Train Acc: 0.9459
Val Loss: 0.1177, Val Acc: 0.9229

Epoch 20/20 [Gated Attention]


Validation: 100%|██████████| 87/87 [00:40<00:00,  2.15it/s]

Train Loss: 0.0799, Train Acc: 0.9459
Val Loss: 0.1157, Val Acc: 0.9218


In [19]:
import torch
import torch.nn as nn
import torchvision.models as models
import timm
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. Define the wrapper to get raw logits (predictions) ---
class FeatureExtractor(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(FeatureExtractor, self).__init__()
        
        # Load EfficientNet
        self.eff = models.efficientnet_b0(weights=None)
        self.eff.classifier[1] = nn.Linear(1280, num_classes)
        self.eff.load_state_dict(torch.load(path_eff))
        
        # Load VGG16
        self.vgg = models.vgg16(weights=None)
        self.vgg.classifier[6] = nn.Linear(4096, num_classes)
        self.vgg.load_state_dict(torch.load(path_vgg))
        
        # Load ViT
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        self.vit.head = nn.Linear(768, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))
        
        # Freeze everything (We only want predictions)
        for model in [self.eff, self.vgg, self.vit]:
            model.eval()
            for param in model.parameters():
                param.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            # Get the logits (probabilities) from each model
            out1 = self.eff(x)
            out2 = self.vgg(x)
            out3 = self.vit(x)
        return out1, out2, out3

# Initialize
extractor = FeatureExtractor(
    num_classes=number_of_classes, 
    path_eff='efficientNetB0.pth', 
    path_vgg='vgg16.pth', 
    path_vit='vision_transformer.pth'
).to(device)

# --- 2. Create the Dataset for the Meta-Learner ---
# We need to run the TRAIN set and VAL set through the models to get inputs for the Logistic Regression

def get_meta_data(loader, model_name=""):
    eff_preds, vgg_preds, vit_preds = [], [], []
    actual_labels = []

    print(f"Extracting features for {model_name}...")
    for images, labels in tqdm(loader):
        images = images.to(device)
        
        o1, o2, o3 = extractor(images)
        
        eff_preds.append(o1.cpu().numpy())
        vgg_preds.append(o2.cpu().numpy())
        vit_preds.append(o3.cpu().numpy())
        actual_labels.append(labels.numpy())
        
    # Stack them: Shape becomes [Total_Images, Num_Classes]
    eff_preds = np.vstack(eff_preds)
    vgg_preds = np.vstack(vgg_preds)
    vit_preds = np.vstack(vit_preds)
    actual_labels = np.concatenate(actual_labels)
    
    # Concatenate all model predictions to make the Meta-Features
    # Shape: [Total_Images, Num_Classes * 3]
    meta_features = np.hstack([eff_preds, vgg_preds, vit_preds])
    
    return meta_features, actual_labels

# Get data
X_train_meta, y_train_meta = get_meta_data(train_loader, "Training Set")
X_val_meta, y_val_meta = get_meta_data(val_loader, "Validation Set")












Extracting features for Training Set...


100%|██████████| 346/346 [02:22<00:00,  2.44it/s]


Extracting features for Validation Set...


100%|██████████| 87/87 [00:43<00:00,  1.98it/s]


In [ ]:
print("\nTraining Meta-Learner (Logistic Regression)...")

# C=1.0 is the regularization strength. 
# Solver 'lbfgs' is a mathematical optimizer that doesn't suffer from "downgrading" like SGD.
meta_learner = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, C=0.1)

# Fit on the predictions of the deep learning models
meta_learner.fit(X_train_meta, y_train_meta)

# --- 3. Final Evaluation ---
print("Evaluating...")
val_predictions = meta_learner.predict(X_val_meta)
val_accuracy = accuracy_score(y_val_meta, val_predictions)

print(f"---------------------------------------")
print(f"Final Ensemble Accuracy: {val_accuracy*100:.2f}%")
print(f"---------------------------------------")

# Check which model the meta-learner trusts the most
# This shows the coefficients (weights) assigned to each model's votes
coeffs = meta_learner.coef_[0] # Look at weights for the first class
n = number_of_classes
print(f"Avg Weight EffNet: {np.mean(np.abs(coeffs[:n])):.4f}")
print(f"Avg Weight VGG16 : {np.mean(np.abs(coeffs[n:2*n])):.4f}")
print(f"Avg Weight ViT   : {np.mean(np.abs(coeffs[2*n:])):.4f}")

d:\Practise\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



Training Meta-Learner (Logistic Regression)...
Evaluating...
---------------------------------------
Final Ensemble Accuracy: 93.52%
---------------------------------------
Avg Weight EffNet: 0.0672
Avg Weight VGG16 : 0.1221
Avg Weight ViT   : 0.1048


In [23]:
class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(CustomEnsemble, self).__init__()

        # --- 1. Load EfficientNet ---
        self.efficientNet = models.efficientnet_b0(weights=None)
        eff_in = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(eff_in, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))
        self.efficientNet.classifier = nn.Identity() # Output: 1280

        # --- 2. Load VGG16 ---
        self.vgg16 = models.vgg16(weights=None)
        vgg_in = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg_in, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))
        self.vgg16.classifier[6] = nn.Identity() # Output: 4096

        # --- 3. Load ViT ---
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_in = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_in, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))
        self.vit.head = nn.Identity() # Output: 768

        # --- 4. Freeze All First ---
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
 
        # --- 5. Unfreeze Specific Layers (Fine Tuning) ---
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True
        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True
        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True        

        # --- 6. Custom Heads (Projectors) ---
        # These compress the varying sizes to a standard 256
        self.eff_head = nn.Sequential(
            nn.Linear(1280, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.3)
        )
        self.vgg_head = nn.Sequential(
            nn.Linear(4096, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.vit_head = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        # --- 7. Fusion Head ---
        # [CRITICAL FIX]: The input is now 256 + 256 + 256, NOT the original sizes
        fusion_input_dim = 256 * 3 

        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256), # Added an extra layer for depth
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Get raw features
        f1 = self.efficientNet(x)
        f2 = self.vgg16(x)
        f3 = self.vit(x)

        # Project to 256 dim each
        f1 = self.eff_head(f1)
        f2 = self.vgg_head(f2)
        f3 = self.vit_head(f3)

        # Concatenate (Size: 768)
        concatenated_features = torch.cat([f1, f2, f3], dim=1)
        
        # Classify
        output = self.fusion_head(concatenated_features)
        
        return output

# Initialize Model
ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    path_vit='vision_transformer.pth'
)
ensemble_model = ensemble_model.to(device)

# ==============================================================================
# DIFFERENTIAL LEARNING RATES IMPLEMENTATION
# ==============================================================================

# 1. Group the parameters
backbone_params = []
head_params = []

# Add Backbones (Only the ones with requires_grad=True will be updated)
backbone_params += [p for p in ensemble_model.efficientNet.parameters() if p.requires_grad]
backbone_params += [p for p in ensemble_model.vgg16.parameters() if p.requires_grad]
backbone_params += [p for p in ensemble_model.vit.parameters() if p.requires_grad]

# Add New Heads (All of these require grad by default)
head_params += list(ensemble_model.eff_head.parameters())
head_params += list(ensemble_model.vgg_head.parameters())
head_params += list(ensemble_model.vit_head.parameters())
head_params += list(ensemble_model.fusion_head.parameters())

# 2. Define Optimizer with Groups
optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5}, # Low LR for pre-trained parts
    {'params': head_params, 'lr': 5e-4}      # Higher LR for new random parts
], weight_decay=1e-3)

# Scheduler (Standard)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

print(f"Optimizer initialized: Backbones (lr=1e-5), Heads (lr=5e-4)")

# ==============================================================================
# TRAINING LOOP (Remains the same)
# ==============================================================================

train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)

Optimizer initialized: Backbones (lr=1e-5), Heads (lr=5e-4)

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.05it/s]


Train Loss: 0.3615, Train Acc: 0.8836
Val Loss: 0.1374, Val Acc: 0.9308

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.05it/s]


Train Loss: 0.1655, Train Acc: 0.9292
Val Loss: 0.1401, Val Acc: 0.9279

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 87/87 [00:43<00:00,  2.02it/s]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.09it/s]


Train Loss: 0.1191, Train Acc: 0.9362
Val Loss: 0.1218, Val Acc: 0.9221

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 87/87 [00:40<00:00,  2.13it/s]

Train Loss: 0.1145, Train Acc: 0.9396
Val Loss: 0.1159, Val Acc: 0.9243


In [24]:
class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(CustomEnsemble, self).__init__()

        
        self.efficientNet = models.efficientnet_b0(weights=None)
        eff_in = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(eff_in, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))
        self.efficientNet.classifier = nn.Identity() 

        
        self.vgg16 = models.vgg16(weights=None)
        vgg_in = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg_in, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))
        self.vgg16.classifier[6] = nn.Identity() 

        
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_in = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_in, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))
        self.vit.head = nn.Identity() 

        # Freeze All First 
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
 
        # Unfreeze Specific Layers 
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True
        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True
        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True        

        # Custom Heads (
       
        self.eff_head = nn.Sequential(
            nn.Linear(1280, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.4)
        )
        self.vgg_head = nn.Sequential(
            nn.Linear(4096, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )
        self.vit_head = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4)
        )

       
        fusion_input_dim = 256 * 3 

        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256), 
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        
        f1 = self.efficientNet(x)
        f2 = self.vgg16(x)
        f3 = self.vit(x)

    
        f1 = self.eff_head(f1)
        f2 = self.vgg_head(f2)
        f3 = self.vit_head(f3)

    
        concatenated_features = torch.cat([f1, f2, f3], dim=1)
        
        
        output = self.fusion_head(concatenated_features)
        
        return output

# Initialize Model
ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    path_vit='vision_transformer.pth'
)
ensemble_model = ensemble_model.to(device)



# Group the parameters
backbone_params = []
head_params = []

# Add Backbones (Only the ones with requires_grad=True will be updated)
backbone_params += [p for p in ensemble_model.efficientNet.parameters() if p.requires_grad]
backbone_params += [p for p in ensemble_model.vgg16.parameters() if p.requires_grad]
backbone_params += [p for p in ensemble_model.vit.parameters() if p.requires_grad]

# Add New Heads (All of these require grad by default)
head_params += list(ensemble_model.eff_head.parameters())
head_params += list(ensemble_model.vgg_head.parameters())
head_params += list(ensemble_model.vit_head.parameters())
head_params += list(ensemble_model.fusion_head.parameters())

# 2. Define Optimizer with Groups
optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5}, # Low LR for pre-trained parts
    {'params': head_params, 'lr': 2e-4}      # Higher LR for new random parts
], weight_decay=1e-3)

# Scheduler (Cosine)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)


train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)


Epoch 1/10


Epoch 1 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.05it/s]


Train Loss: 0.6074, Train Acc: 0.8406
Val Loss: 0.1645, Val Acc: 0.9341

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 87/87 [00:43<00:00,  2.01it/s]


Train Loss: 0.2145, Train Acc: 0.9272
Val Loss: 0.1390, Val Acc: 0.9279

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 87/87 [00:43<00:00,  1.99it/s]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 87/87 [00:44<00:00,  1.95it/s]


Train Loss: 0.1297, Train Acc: 0.9357
Val Loss: 0.1357, Val Acc: 0.9236

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.07it/s]

Train Loss: 0.1235, Train Acc: 0.9416
Val Loss: 0.1290, Val Acc: 0.9265


In [25]:
class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(CustomEnsemble, self).__init__()

        # loading efficientNetB0
        self.efficientNet = models.efficientnet_b0(weights=None)
        efficientNet_features = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(efficientNet_features, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))

        # removing the head to get features
        self.efficientNet.classifier = nn.Identity()

        # loading vgg16
        self.vgg16 = models.vgg16(weights=None)
        vgg16_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg16_features, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))

        self.vgg16.classifier[6] = nn.Identity()

        # loading squeezeNet
        # self.squeezeNet = models.squeezenet1_1(weights=None)
        # self.squeezeNet.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1,1))
        # self.squeezeNet.load_state_dict(torch.load(path_squeeze))


        # loading vision transformer
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_features = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_features, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))

        self.vit.head = nn.Identity()

        # freezing all the layers at first 
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
 

        # Customization
        # unfreezing the last layer of efficientNetB0
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True

        # unfreezing the last layer of vgg16
        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True

        # unfreezing the last layer of vit
        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True        

        # Adding fusion head after adding all features
        total_features = efficientNet_features + vgg16_features + vit_features

        # we have to initialize a neuron layer as a fusion head which will be our classifier
        self.fusion_head = nn.Sequential(
            nn.Linear(total_features, 1024),
            nn.BatchNorm1d(1024),
            nn.Mish(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.Mish(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        f1 = self.efficientNet(x)
        f2 = self.vgg16(x)
        # f3 = self.squeezeNet(x)
        f4 = self.vit(x)

        concatenated_features = torch.cat((f1, f2, f4), dim=1)

        output = self.fusion_head(concatenated_features)
        return output
    


ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    # path_squeeze='squeezeNet_model.pth',
    path_vit='vision_transformer.pth'
)

ensemble_model = ensemble_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, ensemble_model.parameters()), lr=base_learning_rate, weight_decay=1e-3)        
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)




Epoch 1/10


Epoch 1 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.05it/s]


Train Loss: 0.3095, Train Acc: 0.9021
Val Loss: 0.1699, Val Acc: 0.9250

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.07it/s]


Train Loss: 0.1878, Train Acc: 0.9179
Val Loss: 0.1827, Val Acc: 0.9145

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.09it/s]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.08it/s]


Train Loss: 0.1095, Train Acc: 0.9431
Val Loss: 0.1523, Val Acc: 0.9138

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.07it/s]

Train Loss: 0.1098, Train Acc: 0.9422
Val Loss: 0.1496, Val Acc: 0.9152


In [27]:
class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(CustomEnsemble, self).__init__()

        # loading efficientNetB0
        self.efficientNet = models.efficientnet_b0(weights=None)
        efficientNet_features = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(efficientNet_features, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))

        # removing the head to get features
        self.efficientNet.classifier = nn.Identity()

        # loading vgg16
        self.vgg16 = models.vgg16(weights=None)
        vgg16_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg16_features, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))

        self.vgg16.classifier[6] = nn.Identity()

        # loading squeezeNet
        # self.squeezeNet = models.squeezenet1_1(weights=None)
        # self.squeezeNet.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1,1))
        # self.squeezeNet.load_state_dict(torch.load(path_squeeze))


        # loading vision transformer
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_features = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_features, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))

        self.vit.head = nn.Identity()

        # freezing all the layers at first 
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
 

        # Customization
        # unfreezing the last layer of efficientNetB0
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True

        # unfreezing the last layer of vgg16
        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True

        # unfreezing the last layer of vit
        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True        

        # Adding fusion head after adding all features
        total_features = efficientNet_features + vgg16_features + vit_features

        # we have to initialize a neuron layer as a fusion head which will be our classifier
        self.fusion_head = nn.Sequential(
            nn.Linear(total_features, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        f1 = self.efficientNet(x)
        f2 = self.vgg16(x)
        # f3 = self.squeezeNet(x)
        f4 = self.vit(x)

        concatenated_features = torch.cat((f1, f2, f4), dim=1)

        output = self.fusion_head(concatenated_features)
        return output
    


ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    # path_squeeze='squeezeNet_model.pth',
    path_vit='vision_transformer.pth'
)

ensemble_model = ensemble_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, ensemble_model.parameters()), lr=base_learning_rate, weight_decay=1e-3)        
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)




Epoch 1/10


Epoch 1 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.05it/s]


Train Loss: 0.3473, Train Acc: 0.8974
Val Loss: 0.1588, Val Acc: 0.9236

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.09it/s]


Train Loss: 0.1858, Train Acc: 0.9207
Val Loss: 0.1916, Val Acc: 0.9163

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.10it/s]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.11it/s]


Train Loss: 0.1138, Train Acc: 0.9410
Val Loss: 0.1553, Val Acc: 0.9160

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.09it/s]

Train Loss: 0.1112, Train Acc: 0.9426
Val Loss: 0.1503, Val Acc: 0.9203


In [29]:
class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(CustomEnsemble, self).__init__()

        # loading efficientNetB0
        self.efficientNet = models.efficientnet_b0(weights=None)
        efficientNet_features = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(efficientNet_features, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))

        # removing the head to get features
        self.efficientNet.classifier = nn.Identity()

        # loading vgg16
        self.vgg16 = models.vgg16(weights=None)
        vgg16_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg16_features, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))

        self.vgg16.classifier[6] = nn.Identity()

        # loading squeezeNet
        # self.squeezeNet = models.squeezenet1_1(weights=None)
        # self.squeezeNet.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1,1))
        # self.squeezeNet.load_state_dict(torch.load(path_squeeze))


        # loading vision transformer
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_features = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_features, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))

        self.vit.head = nn.Identity()

        # freezing all the layers at first 
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
 

        # Customization
        # unfreezing the last layer of efficientNetB0
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True

        # unfreezing the last layer of vgg16
        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True

        # unfreezing the last layer of vit
        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True        

        # Adding fusion head after adding all features
        total_features = efficientNet_features + vgg16_features + vit_features

        # we have to initialize a neuron layer as a fusion head which will be our classifier
        self.fusion_head = nn.Sequential(
            nn.Linear(total_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        f1 = self.efficientNet(x)
        f2 = self.vgg16(x)
        # f3 = self.squeezeNet(x)
        f4 = self.vit(x)

        concatenated_features = torch.cat((f1, f2, f4), dim=1)

        output = self.fusion_head(concatenated_features)
        return output
    


ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    # path_squeeze='squeezeNet_model.pth',
    path_vit='vision_transformer.pth'
)

ensemble_model = ensemble_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, ensemble_model.parameters()), lr=base_learning_rate, weight_decay=1e-3)        
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)




Epoch 1/40


Epoch 1 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.09it/s]


Train Loss: 0.3566, Train Acc: 0.8868
Val Loss: 0.1922, Val Acc: 0.9073

Epoch 2/40


Epoch 2 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.09it/s]


Train Loss: 0.1932, Train Acc: 0.9185
Val Loss: 0.1813, Val Acc: 0.9181

Epoch 3/40


Epoch 3 Validation: 100%|██████████| 87/87 [00:40<00:00,  2.14it/s]


... (71 outputs left out)

Epoch 39 Validation: 100%|██████████| 87/87 [00:40<00:00,  2.16it/s]


Train Loss: 0.0959, Train Acc: 0.9472
Val Loss: 0.1484, Val Acc: 0.9116

Epoch 40/40


Epoch 40 Validation: 100%|██████████| 87/87 [00:40<00:00,  2.13it/s]

Train Loss: 0.0953, Train Acc: 0.9477
Val Loss: 0.1478, Val Acc: 0.9142


In [30]:
class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(CustomEnsemble, self).__init__()

        
        self.efficientNet = models.efficientnet_b0(weights=None)
        eff_in = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(eff_in, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))
        self.efficientNet.classifier = nn.Identity() 

        
        self.vgg16 = models.vgg16(weights=None)
        vgg_in = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg_in, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))
        self.vgg16.classifier[6] = nn.Identity() 

        
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_in = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_in, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))
        self.vit.head = nn.Identity() 

        # Freeze All First 
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
 
        # Unfreeze Specific Layers 
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True
        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True
        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True        

        # Custom Heads (
       
        self.eff_head = nn.Sequential(
            nn.Linear(1280, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.4)
        )
        self.vgg_head = nn.Sequential(
            nn.Linear(4096, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )
        self.vit_head = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4)
        )

       
        fusion_input_dim = 256 * 3 

        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256), 
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        
        f1 = self.efficientNet(x)
        f2 = self.vgg16(x)
        f3 = self.vit(x)

    
        f1 = self.eff_head(f1)
        f2 = self.vgg_head(f2)
        f3 = self.vit_head(f3)

    
        concatenated_features = torch.cat([f1, f2, f3], dim=1)
        
        
        output = self.fusion_head(concatenated_features)
        
        return output

# Initialize Model
ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    path_vit='vision_transformer.pth'
)
ensemble_model = ensemble_model.to(device)



# Group the parameters
backbone_params = []
head_params = []

# Add Backbones (Only the ones with requires_grad=True will be updated)
backbone_params += [p for p in ensemble_model.efficientNet.parameters() if p.requires_grad]
backbone_params += [p for p in ensemble_model.vgg16.parameters() if p.requires_grad]
backbone_params += [p for p in ensemble_model.vit.parameters() if p.requires_grad]

# Add New Heads (All of these require grad by default)
head_params += list(ensemble_model.eff_head.parameters())
head_params += list(ensemble_model.vgg_head.parameters())
head_params += list(ensemble_model.vit_head.parameters())
head_params += list(ensemble_model.fusion_head.parameters())

# 2. Define Optimizer with Groups
optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5}, # Low LR for pre-trained parts
    {'params': head_params, 'lr': 2e-4}      # Higher LR for new random parts
], weight_decay=1e-3)

# Scheduler (Cosine)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)


train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)


Epoch 1/40


Epoch 1 Validation: 100%|██████████| 87/87 [00:40<00:00,  2.15it/s]


Train Loss: 0.6104, Train Acc: 0.8305
Val Loss: 0.1585, Val Acc: 0.9363

Epoch 2/40


Epoch 2 Validation: 100%|██████████| 87/87 [00:40<00:00,  2.13it/s]


Train Loss: 0.2085, Train Acc: 0.9303
Val Loss: 0.1464, Val Acc: 0.9330

Epoch 3/40


Epoch 3 Validation: 100%|██████████| 87/87 [00:41<00:00,  2.10it/s]


... (59 outputs left out)

Epoch 33 Validation: 100%|██████████| 87/87 [00:42<00:00,  2.05it/s]


Train Loss: 0.0845, Train Acc: 0.9467
Val Loss: 0.1259, Val Acc: 0.9156

Epoch 34/40


Epoch 34 Training:   3%|▎         | 11/346 [00:18<09:36,  1.72s/it] 


KeyboardInterrupt: 

In [37]:
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# ------------------------------------------------------------
# 1. Load your three trained models (same as in your ensemble)
# ------------------------------------------------------------
def load_model_weights(model, path, num_classes, device):
    # 1. EfficientNet Check: Classifier is a Sequential, head is at index [1]
    if isinstance(model, models.EfficientNet):
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        
    # 2. VGG Check: Classifier is a Sequential, head is at index [6]
    elif isinstance(model, models.VGG):
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)
        
    # 3. ViT Check (timm models usually have a .head attribute)
    elif hasattr(model, 'head') and not isinstance(model.head, nn.Identity):
        in_features = model.head.in_features
        model.head = nn.Linear(in_features, num_classes)
    
    # Load weights
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()
    return model

# Instantiate models with original architecture (no custom heads)
effnet = models.efficientnet_b0(weights=None)
vgg16 = models.vgg16(weights=None)
vit = timm.create_model('vit_base_patch16_224', pretrained=False)

# Load your trained weights
effnet = load_model_weights(effnet, 'efficientNetB0.pth', number_of_classes, device)
vgg16 = load_model_weights(vgg16, 'vgg16.pth', number_of_classes, device)
vit   = load_model_weights(vit,   'vision_transformer.pth', number_of_classes, device)

models_list = [effnet, vgg16, vit]

# ------------------------------------------------------------
# 2. Soft Voting Inference on Test Loader
# ------------------------------------------------------------
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Soft Voting"):
        images = images.to(device)
        labels = labels.to(device)
        
        # Collect probabilities from each model
        probs = []
        for model in models_list:
            outputs = model(images)                 # raw logits
            prob = F.softmax(outputs, dim=1)       # probabilities
            probs.append(prob)
        
        # Average probabilities
        avg_prob = torch.mean(torch.stack(probs), dim=0)
        
        # Predicted class
        _, preds = torch.max(avg_prob, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"Soft Voting Ensemble Accuracy: {accuracy:.4f}")

Soft Voting: 100%|██████████| 2437/2437 [01:03<00:00, 38.31it/s]

Soft Voting Ensemble Accuracy: 0.9101


In [38]:
from statistics import mode

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Hard Voting"):
        images = images.to(device)
        labels = labels.to(device)
        
        batch_preds = []
        for model in models_list:
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            batch_preds.append(preds.cpu().numpy())   # shape: (batch_size,)
        
        # Transpose to get (batch_size, n_models)
        batch_preds = np.array(batch_preds).T
        
        # Majority vote per sample
        final_preds = [mode(sample) for sample in batch_preds]
        
        all_preds.extend(final_preds)
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"Hard Voting Ensemble Accuracy: {accuracy:.4f}")

Hard Voting: 100%|██████████| 2437/2437 [01:07<00:00, 36.03it/s]

Hard Voting Ensemble Accuracy: 0.9110


In [44]:
class CustomEnsemble(nn.Module):
    def __init__(self, num_classes, path_eff, path_vgg, path_vit):
        super(CustomEnsemble, self).__init__()

        # loading efficientNetB0
        self.efficientNet = models.efficientnet_b0(weights=None)
        efficientNet_features = self.efficientNet.classifier[1].in_features
        self.efficientNet.classifier[1] = nn.Linear(efficientNet_features, num_classes)
        self.efficientNet.load_state_dict(torch.load(path_eff))

        # removing the head to get features
        self.efficientNet.classifier = nn.Identity()

        # loading vgg16
        self.vgg16 = models.vgg16(weights=None)
        vgg16_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(vgg16_features, num_classes)
        self.vgg16.load_state_dict(torch.load(path_vgg))

        self.vgg16.classifier[6] = nn.Identity()

        # loading squeezeNet
        # self.squeezeNet = models.squeezenet1_1(weights=None)
        # self.squeezeNet.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1,1))
        # self.squeezeNet.load_state_dict(torch.load(path_squeeze))


        # loading vision transformer
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False)
        vit_features = self.vit.head.in_features
        self.vit.head = nn.Linear(vit_features, num_classes)
        self.vit.load_state_dict(torch.load(path_vit))

        self.vit.head = nn.Identity()

        # freezing all the layers at first 
        for model in [self.efficientNet, self.vgg16, self.vit]:
            for param in model.parameters():
                param.requires_grad = False
 

        # Customization
        # unfreezing the last layer of efficientNetB0
        for param in self.efficientNet.features[7:].parameters():
            param.requires_grad = True

        # unfreezing the last layer of vgg16
        for param in self.vgg16.features[24:].parameters():
            param.requires_grad = True

        # unfreezing the last layer of vit
        for param in self.vit.blocks[-1].parameters():
            param.requires_grad = True        

        # Adding fusion head after adding all features
        total_features = efficientNet_features + vgg16_features + vit_features

        # we have to initialize a neuron layer as a fusion head which will be our classifier
        self.fusion_head = nn.Sequential(
            nn.Linear(total_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.7),   # very high dropout
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        f1 = self.efficientNet(x)
        f2 = self.vgg16(x)
        # f3 = self.squeezeNet(x)
        f4 = self.vit(x)

        concatenated_features = torch.cat((f1, f2, f4), dim=1)

        output = self.fusion_head(concatenated_features)
        return output
    


ensemble_model = CustomEnsemble(
    num_classes=number_of_classes,
    path_eff='efficientNetB0.pth',
    path_vgg='vgg16.pth',
    # path_squeeze='squeezeNet_model.pth',
    path_vit='vision_transformer.pth'
)

ensemble_model = ensemble_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, ensemble_model.parameters()), lr=base_learning_rate, weight_decay=1e-3)        
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    ensemble_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ensemble_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    ensemble_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = ensemble_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step(val_loss)




Epoch 1/5


Epoch 1 Validation: 100%|██████████| 77/77 [00:39<00:00,  1.96it/s]


Train Loss: 0.4095, Train Acc: 0.8859
Val Loss: 0.2044, Val Acc: 0.9130

Epoch 2/5


Epoch 2 Validation: 100%|██████████| 77/77 [00:41<00:00,  1.85it/s]


Train Loss: 0.2262, Train Acc: 0.9215
Val Loss: 0.1943, Val Acc: 0.9187

Epoch 3/5


Epoch 3 Validation: 100%|██████████| 77/77 [00:38<00:00,  2.01it/s]


Train Loss: 0.2017, Train Acc: 0.9244
Val Loss: 0.1878, Val Acc: 0.9208

Epoch 4/5


Epoch 4 Validation: 100%|██████████| 77/77 [00:38<00:00,  2.02it/s]


Train Loss: 0.1971, Train Acc: 0.9221
Val Loss: 0.1910, Val Acc: 0.9142

Epoch 5/5


Epoch 5 Validation: 100%|██████████| 77/77 [00:40<00:00,  1.92it/s]

Train Loss: 0.1977, Train Acc: 0.9207
Val Loss: 0.2267, Val Acc: 0.9031
